# Systematic pairs trading: S&P 500 sector screener

A separate research extension of `microestructuraprecios_ieb_v1.ipynb`. The original is unchanged.

**Workflow:** current constituent snapshot → Yahoo adjusted prices → formation tests → raw p-value filter (uncorrected) → validation leaderboard → freeze up to five pairs → final holdout diagnostics.

Run the cells in order in Jupyter or Google Colab. This notebook embeds its implementation and does not require the accompanying Python file. The defaults scan every within-sector pair. To start with economically closer businesses, set `group_by="subindustry"`. All-market comparisons are available with `group_by="all"`.

This is a research screener, not a live trading system. Current membership and current sector classifications cause survivorship/selection bias in historical results. A final chronological holdout does not remove this universe bias.


In [ ]:
%pip install -q "numpy>=1.26,<3" "pandas>=2.2,<4" "statsmodels>=0.14,<0.16" "matplotlib>=3.8,<4" "yfinance>=0.2.54,<2" "requests>=2.31,<3" "lxml>=5,<7"


## What changes from the teaching notebook?

- Training hedge estimation and Engle–Granger are retained. Plain residual ADF p-values are not counted as an independent cointegration confirmation: estimated residuals need cointegration-specific critical values.
- Formation, validation and final holdout use common calendar boundaries. No holdout information enters the leaderboard. Coefficients remain frozen from formation to make the first version auditable.
- Every economic candidate pair belongs to the global test family, including failed or untested pairs assigned p=1. The default uses raw Engle–Granger p-values without correction, as requested. This is exploratory screening: a 5% per-test cutoff does not control false discoveries across the search. Set `fdr_method="fdr_bh"` or `"fdr_by"` to restore optional correction.
- Unit-root ADF checks are compatibility diagnostics, not proof of I(1). Half-life and split-half beta stability are heuristic gates, not independent formal significance tests. Do not interpret passing several gates as multiplying confidence.
- A close-t signal fills at close t+1 and earns its first price P&L through close t+2. Equity is capital plus marked-to-market dollar P&L minus costs, with percentage drawdown from a peak including starting capital.
- Missing prices are never forward-filled. Formation failures are audited; unavailable selected holdouts are reported without replacing the pair with a future winner.


## Implementation
The next cell contains the reusable engine, also supplied separately as `pairs_screener.py`.

In [ ]:
"""Sector-based pairs research. No broker integration or live orders.

The companion notebook embeds this module and can run independently.
"""
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from io import StringIO
import hashlib
import importlib.metadata
import json
import time
import warnings
import inspect
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.stats.multitest import multipletests


def constant_design(x):
    return np.column_stack([np.ones(len(x)), x])


def adf_pvalue(series, config):
    # Statsmodels 0.15 added an explicit return-type flag; retain 0.14 compatibility.
    options = {"result_object": False} if "result_object" in inspect.signature(adfuller).parameters else {}
    return adfuller(series, maxlag=config.maxlag, autolag="AIC", **options)[1]


@dataclass(frozen=True)
class Config:
    start: str = "2020-01-01"
    end: str = datetime.now(timezone.utc).date().isoformat()  # exclusive
    formation_fraction: float = 0.60
    validation_fraction: float = 0.20
    group_by: str = "sector"  # sector, subindustry, or all
    sectors: tuple = ()  # empty means all sectors
    top_n: int = 5
    max_pairs_per_sector: int = 2
    disjoint_tickers: bool = True
    fdr_method: str = "none"  # raw p-values; optional fdr_bh or fdr_by
    fdr_alpha: float = 0.05
    min_formation: int = 504
    min_evaluation: int = 126
    min_dollar_volume: float = 20_000_000
    min_price: float = 5.0
    min_return_correlation: float = 0.40
    min_beta: float = 0.10
    max_beta: float = 5.0
    min_half_life: float = 2.0
    max_half_life: float = 60.0
    max_beta_relative_change: float = 0.50
    maxlag: int = 5  # AIC selects among 0..5; fixed before running
    lookback: int = 60
    entry_z: float = 2.0
    exit_z: float = 0.5
    stop_z: float = 4.0
    max_holding: int = 60
    cost_bps: float = 5.0  # commission + slippage per traded dollar, each leg
    borrow_bps_year: float = 100.0
    capital: float = 100_000.0  # per pair
    gross_at_entry: float = 1.0
    min_trades: int = 5
    max_validation_drawdown: float = 0.20
    rank_by: str = "composite"  # composite, sharpe, pnl, statistical

    def __post_init__(self):
        if not (0 < self.formation_fraction < 1 and 0 < self.validation_fraction < 1
                and self.formation_fraction + self.validation_fraction < 1):
            raise ValueError("Formation/validation fractions must leave a holdout.")
        if not (0 <= self.exit_z < self.entry_z < self.stop_z):
            raise ValueError("Require 0 <= exit_z < entry_z < stop_z.")
        if self.group_by not in {"sector", "subindustry", "all"}:
            raise ValueError("Invalid grouping")
        if self.rank_by not in {"composite", "sharpe", "pnl", "statistical"}:
            raise ValueError("Invalid ranking")
        if self.fdr_method not in {"none", "fdr_by", "fdr_bh"}:
            raise ValueError("Use none, fdr_by or fdr_bh")
        if not (0 < self.fdr_alpha < 1 and self.capital > 0 and 0 < self.gross_at_entry <= 1):
            raise ValueError("Invalid significance, capital or leverage")
        if min(self.top_n, self.max_pairs_per_sector, self.lookback - 2,
               self.min_formation, self.min_evaluation, self.max_holding) < 1:
            raise ValueError("Invalid count/window")
        if min(self.cost_bps, self.borrow_bps_year) < 0:
            raise ValueError("Costs cannot be negative")


def get_universe(directory, csv_path=None):
    """Fetch current membership, or explicitly supplied snapshot; never silent fallback."""
    import requests
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    if csv_path:
        universe = pd.read_csv(csv_path, dtype={"cik": str})
        source = str(csv_path)
    else:
        source = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
        response = requests.get(source, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        response.raise_for_status()
        tables = pd.read_html(StringIO(response.text))
        universe = next(t for t in tables if {"Symbol", "GICS Sector"}.issubset(t.columns))
        universe = universe.rename(columns={"Symbol": "ticker", "Security": "name",
            "GICS Sector": "sector", "GICS Sub-Industry": "subindustry", "CIK": "cik"})
        if not 450 <= len(universe) <= 550:
            raise ValueError("Unexpected constituent count; inspect the source table.")
    required = {"ticker", "name", "sector", "subindustry", "cik"}
    if not required.issubset(universe.columns) or universe[list(required)].isna().any().any():
        raise ValueError(f"Universe needs nonmissing columns: {sorted(required)}")
    universe["ticker"] = universe.ticker.str.replace(".", "-", regex=False)
    universe["cik"] = universe.cik.astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10)
    universe = universe.sort_values("ticker").drop_duplicates("ticker").reset_index(drop=True)
    universe.to_csv(directory / "universe_snapshot.csv", index=False)
    (directory / "universe_source.json").write_text(json.dumps({"source": source,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "note": "Current membership and classification; NOT point-in-time historical membership."}, indent=2))
    return universe


def download_prices(universe, config, directory, batch_size=40):
    """Download once per ticker in batches; retry failures individually and audit coverage."""
    import yfinance as yf
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    tickers = sorted(set(universe.ticker) | {"SPY"})  # common session calendar
    data = {}
    failures = {}
    fields = ["Adj Close", "Close", "Volume"]

    def ingest(raw, symbols):
        if raw is None or raw.empty:
            return
        for symbol in symbols:
            try:
                block = raw.xs(symbol, level=1, axis=1) if isinstance(raw.columns, pd.MultiIndex) else raw
                block = block.reindex(columns=fields).copy()
                block.index = pd.to_datetime(block.index).tz_localize(None).normalize()
                block = block.loc[~block.index.duplicated()].sort_index()
                if block["Adj Close"].notna().any():
                    data[symbol] = block
            except (KeyError, ValueError) as exc:
                failures[symbol] = str(exc)

    for first in range(0, len(tickers), batch_size):
        batch = tickers[first:first + batch_size]
        try:
            ingest(yf.download(batch, start=config.start, end=config.end, auto_adjust=False,
                group_by="column", threads=4, progress=False, timeout=25), batch)
        except Exception as exc:
            failures.update({t: str(exc) for t in batch})
        print(f"Downloaded batch {first // batch_size + 1}: {len(data)}/{len(tickers)} symbols")
    for ticker in sorted(set(tickers) - set(data)):
        for attempt in range(2):
            try:
                ingest(yf.download([ticker], start=config.start, end=config.end,
                    auto_adjust=False, progress=False, threads=False, timeout=25), [ticker])
                if ticker in data:
                    break
            except Exception as exc:
                failures[ticker] = str(exc)
            time.sleep(attempt + 1)
    if "SPY" not in data:
        raise RuntimeError("SPY calendar unavailable. Retry the download; do not invent sessions.")
    calendar = data["SPY"]["Adj Close"].dropna().index
    prices = pd.DataFrame({t: data[t]["Adj Close"] for t in data}, index=calendar)
    dollars = pd.DataFrame({t: data[t]["Close"] * data[t]["Volume"] for t in data}, index=calendar)
    prices = prices.reindex(columns=universe.ticker).where(lambda x: x > 0)
    dollars = dollars.reindex(columns=universe.ticker)
    audit = pd.DataFrame([{"ticker": t, "observations": int(prices[t].notna().sum()),
        "missing_sessions": int(prices[t].isna().sum()),
        "download_status": "ok" if t in data else "failed",
        "error": failures.get(t, "") if t not in data else ""} for t in universe.ticker])
    prices.to_csv(directory / "adjusted_close.csv")
    dollars.to_csv(directory / "dollar_volume.csv")
    audit.to_csv(directory / "download_audit.csv", index=False)
    return prices, dollars, audit


def split_dates(prices, config):
    index = prices.index
    if not index.is_unique or not index.is_monotonic_increasing:
        raise ValueError("Price dates must be unique and sorted")
    a = int(len(index) * config.formation_fraction)
    b = int(len(index) * (config.formation_fraction + config.validation_fraction))
    if a < config.min_formation or min(b-a, len(index)-b) < config.min_evaluation:
        raise ValueError("Not enough sessions for formation, validation and holdout")
    return index[:a], index[a:b], index[b:]


def fit_pair(logs, config):
    y, x = logs.iloc[:, 0].to_numpy(), logs.iloc[:, 1].to_numpy()
    alpha, beta = np.linalg.lstsq(constant_design(x), y, rcond=None)[0]
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        stat, pvalue, _ = coint(y, x, trend="c", maxlag=config.maxlag, autolag="aic")
    residual = y - alpha - beta*x
    phi = np.linalg.lstsq(constant_design(residual[:-1]), residual[1:], rcond=None)[0][1]
    half_life = -np.log(2)/np.log(phi) if 0 < phi < 1 else np.inf
    half = len(logs)//2
    betas = [np.linalg.lstsq(constant_design(x[s]), y[s], rcond=None)[0][1]
             for s in [slice(None, half), slice(half, None)]]
    change = abs(betas[1]-betas[0])/max(abs(beta), 1e-8)
    # Nonfinite statistics can indicate near-perfect collinearity; reject rather than promote.
    valid = np.isfinite(stat) and np.isfinite(pvalue) and residual.std() > 1e-8
    return {"alpha": alpha, "beta": beta, "eg_stat": stat,
        "pvalue": float(pvalue) if valid else 1.0, "half_life": half_life,
        "beta_relative_change": change, "valid_test": bool(valid),
        "warnings": " | ".join(str(w.message) for w in captured)}


def screen_pairs(prices, dollars, universe, formation, config):
    """One predeclared alphabetical orientation; optional correction of the pair family.

    Pairs not tested (data/issuer failures) stay in the family with p=1.
    Raw mode applies no multiple-testing correction. Other gates remain identical.
    """
    u = universe[universe.sector.isin(config.sectors)] if config.sectors else universe.copy()
    p = prices.reindex(index=formation, columns=u.ticker)
    d = dollars.reindex(index=formation, columns=u.ticker)
    ticker_rows = []
    for t in u.ticker:
        complete = p[t].notna().all() and (p[t] > 0).all()
        liquid = d[t].notna().all() and d[t].median() >= config.min_dollar_volume
        row = {"ticker": t, "complete_formation": bool(complete),
            "median_dollar_volume": d[t].median(), "liquid": bool(liquid),
            "level_adf_p": np.nan, "diff_adf_p": np.nan, "error": ""}
        if complete:
            try:
                logs = np.log(p[t])
                row["level_adf_p"] = adf_pvalue(logs, config)
                row["diff_adf_p"] = adf_pvalue(logs.diff().dropna(), config)
            except ValueError as exc:
                row["error"] = str(exc)
        row["eligible"] = bool(complete and liquid and p[t].median() >= config.min_price
            and row["level_adf_p"] > .05 and row["diff_adf_p"] < .05)
        ticker_rows.append(row)
    ta = pd.DataFrame(ticker_rows).set_index("ticker")
    metadata = u.set_index("ticker")
    groups = [("all", u)] if config.group_by == "all" else u.groupby(config.group_by, sort=True)
    rows = []
    for group_name, group in groups:
        for y, x in combinations(sorted(group.ticker), 2):
            row = {"y": y, "x": x, "pair": f"{y}/{x}", "group": group_name,
                "sector": metadata.loc[y, "sector"], "pvalue": 1.0, "reason": "",
                "valid_test": False, "corr": np.nan, "alpha": np.nan, "beta": np.nan,
                "half_life": np.nan, "beta_relative_change": np.nan, "warnings": ""}
            if metadata.loc[y, "cik"] == metadata.loc[x, "cik"]:
                row["reason"] = "same_issuer"
            elif not (ta.loc[y, "complete_formation"] and ta.loc[x, "complete_formation"]):
                row["reason"] = "incomplete_formation"
            else:
                try:
                    logs = np.log(p[[y, x]])
                    row.update(fit_pair(logs, config))
                    row["corr"] = logs.diff().corr().iloc[0, 1]
                except (ValueError, np.linalg.LinAlgError) as exc:
                    row["reason"] = f"test_error: {exc}"
            rows.append(row)
        print(f"Screened {group_name}; cumulative pair family: {len(rows)}")
    if not rows:
        return pd.DataFrame(columns=["pair", "eligible", "reason"]), ta.reset_index()
    table = pd.DataFrame(rows)
    table["qvalue"] = (np.nan if config.fdr_method == "none" else
        multipletests(table.pvalue, alpha=config.fdr_alpha, method=config.fdr_method)[1])
    table["selection_pvalue"] = table.pvalue if config.fdr_method == "none" else table.qvalue
    table["correction_method"] = config.fdr_method
    for i, row in table.iterrows():
        reasons = [row.reason] if row.reason else []
        if not row.valid_test: reasons.append("invalid_or_unavailable_test")
        if row.selection_pvalue > config.fdr_alpha:
            reasons.append("raw_pvalue" if config.fdr_method == "none" else "FDR")
        if not (ta.loc[row.y, "eligible"] and ta.loc[row.x, "eligible"]): reasons.append("ticker_gate")
        if not row["corr"] >= config.min_return_correlation: reasons.append("correlation")
        if not config.min_beta <= row.beta <= config.max_beta: reasons.append("beta")
        if not config.min_half_life <= row.half_life <= config.max_half_life: reasons.append("half_life")
        if not row.beta_relative_change <= config.max_beta_relative_change: reasons.append("beta_instability")
        table.loc[i, "reason"] = ";".join(reasons)
    table["eligible"] = table.reason.eq("")
    return table, ta.reset_index()


def backtest(prices, model, dates, config, cost_multiplier=1.0):
    """Signal at close t; fill close t+1; first price P&L through close t+2.

    Adjusted-price units held fixed within a trade. At entry dollar weights are
    (1,-beta)/(1+abs(beta)), scaled to current equity. Borrow accrues on actual
    short notional using calendar days. Final close is a scheduled liquidation.
    """
    cols = [model["y"], model["x"]]
    end_loc = prices.index.get_loc(dates[-1])
    first = prices.index.get_loc(dates[0])
    p = prices.loc[:, cols].iloc[max(0, first-config.lookback-2):end_loc+1].copy()
    if p.isna().any().any() or (p <= 0).any().any():
        raise ValueError("Missing/nonpositive evaluation or warmup prices: period cannot be valued safely")
    s = np.log(p.iloc[:, 0])-model["alpha"]-model["beta"]*np.log(p.iloc[:, 1])
    z = (s-s.rolling(config.lookback).mean().shift(1))/s.rolling(config.lookback).std().shift(1)
    lag_z = z.shift(1)
    qty = np.zeros(2)
    equity = config.capital
    state = 0
    age = 0
    entries = []
    trade = None
    records = []
    rate = config.cost_bps/10000*cost_multiplier
    for date in dates:
        loc = p.index.get_loc(date)
        current = p.iloc[loc].to_numpy()
        previous = p.iloc[loc-1].to_numpy()
        old_equity = equity
        gross_pnl = float(qty @ (current-previous))
        days = (date-p.index[loc-1]).days
        borrow = float(np.maximum(-qty*previous, 0).sum())*config.borrow_bps_year/10000*days/365
        equity += gross_pnl-borrow
        age = age+1 if state else 0
        value = lag_z.loc[date]
        new_state = state
        reason = ""
        if state:
            if date == dates[-1]: new_state, reason = 0, "period_end"
            elif not np.isfinite(value): new_state, reason = 0, "invalid_signal"
            elif abs(value) >= config.stop_z: new_state, reason = 0, "z_stop"
            elif age >= config.max_holding: new_state, reason = 0, "time_stop"
            elif (state == 1 and value >= -config.exit_z) or (state == -1 and value <= config.exit_z):
                new_state, reason = 0, "mean_reversion"
        elif date != dates[-1] and np.isfinite(value) and config.entry_z <= abs(value) < config.stop_z:
            new_state = -1 if value > 0 else 1
        if equity <= 0:
            raise ValueError("Equity exhausted; backtest invalid")
        cost = 0.0
        delta = np.zeros(2)
        if new_state != state:
            if new_state:
                weights = np.array([1.0, -model["beta"]])/(1+abs(model["beta"]))
                new_qty = new_state*weights*equity*config.gross_at_entry/current
                trade = {"entry": date, "direction": new_state, "equity_before": equity}
                age = 0
            else:
                new_qty = np.zeros(2)
            delta = new_qty-qty
            cost = float(np.abs(delta*current).sum())*rate
            equity -= cost
            if not new_state:
                entries.append({**trade, "exit": date, "sessions": age,
                    "net_pnl": equity-trade["equity_before"], "exit_reason": reason})
                trade = None
            qty, state = new_qty, new_state
        if equity <= 0:
            raise ValueError("Equity exhausted after costs")
        records.append({"date": date, "equity": equity, "net_return": equity/old_equity-1,
            "gross_pnl": gross_pnl, "cost": cost, "borrow": borrow, "position": state,
            "spread": s.loc[date], "zscore": z.loc[date], "signal_used": value,
            "price_y": current[0], "price_x": current[1], "delta_y": delta[0], "delta_x": delta[1],
            "gross_exposure": float(np.abs(qty*current).sum()/equity)})
    bt = pd.DataFrame(records).set_index("date")
    bt["pnl"] = bt.equity-config.capital
    bt["drawdown"] = bt.equity/bt.equity.cummax().clip(lower=config.capital)-1
    return bt, pd.DataFrame(entries, columns=["entry", "direction", "equity_before", "exit", "sessions", "net_pnl", "exit_reason"])


def metrics(bt, trades, config):
    r = bt.net_return
    vol = r.std(ddof=1)
    sharpe = r.mean()/vol*np.sqrt(252) if vol > 1e-12 else np.nan
    blocks = [r.iloc[idx] for idx in np.array_split(np.arange(len(r)), 3)]
    return {"sharpe": sharpe, "pnl": bt.equity.iloc[-1]-config.capital,
        "return": bt.equity.iloc[-1]/config.capital-1,
        "cagr": (bt.equity.iloc[-1]/config.capital)**(252/len(r))-1,
        "max_drawdown": bt.drawdown.min(), "trades": len(trades),
        "win_rate": (trades.net_pnl > 0).mean() if len(trades) else np.nan,
        "positive_blocks": np.mean([((1+b).prod()-1) > 0 for b in blocks]),
        "costs": bt.cost.sum()+bt.borrow.sum(), "max_gross_exposure": bt.gross_exposure.max()}


def rank_validation(screen, prices, validation, config):
    rows = []
    for model in screen.loc[screen.eligible].to_dict("records"):
        row = dict(model)
        try:
            bt, trades = backtest(prices, model, validation, config)
            stress, st = backtest(prices, model, validation, config, cost_multiplier=2)
            m = metrics(bt, trades, config)
            row.update({"val_"+k: v for k, v in m.items()})
            row["stress_pnl"] = stress.equity.iloc[-1]-config.capital
            row["validation_status"] = "ok"
            row["qualified"] = bool(np.isfinite(m["sharpe"]) and m["sharpe"] > 0
                and m["pnl"] > 0 and row["stress_pnl"] > 0
                and m["trades"] >= config.min_trades
                and m["max_drawdown"] >= -config.max_validation_drawdown)
        except ValueError as exc:
            row.update(qualified=False, validation_status=str(exc))
        rows.append(row)
    if not rows:
        return pd.DataFrame(columns=["pair", "qualified", "score"])
    board = pd.DataFrame(rows)
    board["score"] = np.nan
    ok = board.qualified
    if ok.any():
        q = board.loc[ok]
        board.loc[ok, "score"] = (.50*q.val_sharpe.rank(pct=True)
            + .20*q.val_max_drawdown.rank(pct=True)
            + .20*q.val_positive_blocks.rank(pct=True)
            + .10*(-q.selection_pvalue).rank(pct=True))
    sort = {"composite": "score", "sharpe": "val_sharpe", "pnl": "val_pnl", "statistical": "selection_pvalue"}[config.rank_by]
    if sort not in board:
        board[sort] = np.nan
    return board.sort_values(["qualified", sort, "pair"], ascending=[False, config.rank_by == "statistical", True], na_position="last").reset_index(drop=True)


def select_pairs(board, config):
    selected, used, sectors = [], set(), {}
    for row in board.loc[board.qualified].to_dict("records"):
        if config.disjoint_tickers and ({row["y"], row["x"]} & used): continue
        if sectors.get(row["sector"], 0) >= config.max_pairs_per_sector: continue
        selected.append(row)
        used.update([row["y"], row["x"]])
        sectors[row["sector"]] = sectors.get(row["sector"], 0)+1
        if len(selected) == config.top_n: break
    return pd.DataFrame(selected, columns=board.columns)


def plot_pair(bt, model, config, destination=None):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(3, 2, figsize=(14, 11), constrained_layout=True)
    fig.suptitle(f"{model['pair']} | FINAL HOLDOUT | selected using validation only", fontsize=14)
    for ax, leg in zip(axes[0], ["y", "x"]):
        ax.plot(bt.index, bt["price_"+leg], linewidth=1)
        for mask, marker, color, label in [(bt["delta_"+leg]>0, "^", "green", "Buy"),
                                          (bt["delta_"+leg]<0, "v", "red", "Sell")]:
            ax.scatter(bt.index[mask], bt.loc[mask, "price_"+leg], marker=marker, c=color, s=25, label=label)
        ax.set_title(model[leg]+" adjusted price / execution events")
        ax.legend()
    axes[1, 0].plot(bt.index, bt.spread)
    axes[1, 0].set_title("Log-price residual (formation coefficients frozen)")
    axes[1, 1].plot(bt.index, bt.zscore)
    for threshold in [-config.stop_z, -config.entry_z, -config.exit_z, config.exit_z, config.entry_z, config.stop_z]:
        axes[1, 1].axhline(threshold, ls="--", alpha=.4)
    axes[1, 1].set_title("Z-score (fills use previous session's value)")
    axes[2, 0].plot(bt.index, bt.equity, label="Net equity ($)")
    axes[2, 0].axhline(config.capital, color="gray", ls=":")
    axes[2, 0].set_title(f"Net P&L: USD {bt.pnl.iloc[-1]:,.0f} on USD {config.capital:,.0f}")
    axes[2, 0].legend()
    axes[2, 1].fill_between(bt.index, bt.drawdown*100, 0, alpha=.5)
    axes[2, 1].set_title("Drawdown (%)")
    for ax in axes.flat:
        ax.grid(alpha=.2)
        ax.tick_params(axis="x", rotation=25)
    if destination:
        fig.savefig(destination, dpi=140)
    return fig


def run_research(prices, dollars, universe, config, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    formation, validation, holdout = split_dates(prices, config)
    screen, ticker_audit = screen_pairs(prices, dollars, universe, formation, config)
    screen.to_csv(directory / "formation_screen.csv", index=False)
    ticker_audit.to_csv(directory / "ticker_audit.csv", index=False)
    board = rank_validation(screen, prices, validation, config)
    board.to_csv(directory / "validation_leaderboard.csv", index=False)
    selected = select_pairs(board, config)
    selected.to_csv(directory / "selected_before_holdout.csv", index=False)
    # Freeze selection on disk BEFORE touching any selected pair's holdout performance.
    reports, curves = [], {}
    for model in selected.to_dict("records"):
        try:
            bt, trades = backtest(prices, model, holdout, config)
            name = model["pair"].replace("/", "_")
            bt.to_csv(directory / f"{name}_holdout_daily.csv")
            trades.to_csv(directory / f"{name}_holdout_trades.csv", index=False)
            reports.append({"pair": model["pair"], "status": "ok", **metrics(bt, trades, config)})
            curves[model["pair"]] = bt
        except ValueError as exc:
            # No replacement/reranking using future availability or performance.
            reports.append({"pair": model["pair"], "status": "unavailable: "+str(exc)})
    report = pd.DataFrame(reports, columns=None if reports else ["pair", "status"])
    report.to_csv(directory / "holdout_report.csv", index=False)
    versions = {}
    for package in ["numpy", "pandas", "statsmodels", "yfinance", "matplotlib"]:
        try: versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: versions[package] = "unavailable"
    manifest = {"config": asdict(config), "versions": versions,
        "run_utc": datetime.now(timezone.utc).isoformat(),
        "periods": {k: [str(v[0].date()), str(v[-1].date()), len(v)] for k, v in
            [("formation", formation), ("validation", validation), ("holdout", holdout)]},
        "family_size": len(screen), "selected": list(selected.pair),
        "prices_sha256": hashlib.sha256(prices.to_csv().encode()).hexdigest(),
        "universe_sha256": hashlib.sha256(universe.to_csv(index=False).encode()).hexdigest(),
        "limitation": "Current-membership research; not survivorship-free or an executable broker backtest."}
    (directory / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
    print(f"Family: {len(screen):,}; formation eligible: {int(screen.eligible.sum())}; selected: {len(selected)}")
    return {"screen": screen, "leaderboard": board, "selected": selected,
            "holdout": report, "curves": curves, "manifest": manifest}


## Your settings

The last available completed daily session is used (`end` is exclusive). The default split is 60% formation, 20% validation and 20% final holdout. These are configurable research choices, not optimized parameters.

Default gates: raw EG p-value ≤ 5%; positive beta 0.1–5; half-life 2–60 sessions; split-half beta change ≤ 50%; formation return correlation ≥ 0.4; median daily dollar volume ≥ $20m. Validation needs at least five closed trades, positive net Sharpe/P&L, drawdown no worse than 20%, and positive P&L at doubled transaction costs. An empty shortlist is a valid outcome. Do not loosen gates after seeing holdout results.


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

CONFIG = Config(
    group_by="sector",          # "subindustry" is a narrower economic universe
    sectors=(),                 # e.g. ("Consumer Staples", "Utilities")
    top_n=5,
    rank_by="composite",         # "sharpe", "pnl", "statistical"
    fdr_method="none",          # raw p-values; optional "fdr_bh" or "fdr_by"
    fdr_alpha=0.05,              # cutoff applied to raw p-values in this mode
    disjoint_tickers=True,
    max_pairs_per_sector=2,
)
RUN_DIR = Path("pairs_results") / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=True)
UNIVERSE_CSV = None  # optional explicit snapshot with ticker,name,sector,subindustry,cik
display(pd.Series(asdict(CONFIG)))


## Download current constituents and Yahoo prices

Membership and GICS classifications come from the referenced public constituent table; prices and volume come from Yahoo Finance. No fixed constituent list is embedded. The snapshot retrieval time is recorded; verify membership with an authoritative licensed source before deployment. Multiple share classes mean the number of symbols need not be exactly 500.

Downloads are batched and failed symbols are retried. Raw adjusted closes, dollar volume and failures are saved. Reuse a saved dataset by loading those CSV files explicitly instead of rerunning this cell. No stale cache is substituted silently.


In [ ]:
universe = get_universe(RUN_DIR, UNIVERSE_CSV)
if CONFIG.sectors:
    universe = universe[universe.sector.isin(CONFIG.sectors)].copy()
if universe.empty:
    raise ValueError("No constituents match the requested sectors")
display(universe.groupby(CONFIG.group_by if CONFIG.group_by != "all" else "sector").size().rename("symbols"))
prices, dollars, download_audit = download_prices(universe, CONFIG, RUN_DIR)
display(download_audit[download_audit.download_status != "ok"])
formation, validation, holdout = split_dates(prices, CONFIG)
display(pd.DataFrame({k: {"start": v[0], "end": v[-1], "sessions": len(v)} for k, v in
    [("formation", formation), ("validation", validation), ("holdout", holdout)]}).T)


## Screen, rank and freeze the shortlist

Composite score: **50% validation Sharpe percentile + 20% drawdown percentile + 20% positive-validation-block fraction percentile + 10% statistical-strength percentile**. All ranks are among qualified pairs; a higher score is better. Three contiguous validation blocks describe consistency, not three independent out-of-sample experiments. `statistical` ranking sorts the selected p-value measure ascending (raw by default); other choices sort descending. Sharpe assumes zero cash benchmark; financing/cash interest is not modeled.

The shortlist is greedily selected in leaderboard order, with no repeated ticker and at most two pairs per sector by default. These limits reduce concentration but do not guarantee independent pair returns or market neutrality. The code saves selection before evaluating the final holdout. Changing ranking settings after looking at holdout outcomes consumes the holdout.


In [ ]:
results = run_research(prices, dollars, universe, CONFIG, RUN_DIR)
board = results["leaderboard"]
visible = [c for c in ["pair", "sector", "qualified", "score", "pvalue", "selection_pvalue", "correction_method", "half_life",
    "val_sharpe", "val_pnl", "val_max_drawdown", "val_trades", "val_win_rate",
    "val_positive_blocks", "stress_pnl", "validation_status"] if c in board]
display(board[visible].head(25))
print("Full leaderboard and rejection audit saved to:", RUN_DIR.resolve())
print("Selected BEFORE holdout:", results["selected"].pair.tolist())
if results["selected"].empty:
    print("No qualifying pairs. Inspect the rejection audit; no top-five winners are forced.")
    print("Formation audit below: smallest RAW p-values, not qualified trading recommendations.")
    audit_columns = ["pair", "pvalue", "selection_pvalue", "correction_method", "eligible", "reason"]
    display(results["screen"].sort_values("pvalue")[audit_columns].head(10))


## Final holdout: keep the selection order, including losers

This table reports subsequent performance of the validation-selected pairs. It is deliberately not sorted by final P&L. Each pair has its own $100,000 capital allocation; these are individual strategies, not a portfolio backtest. A failure to value a pair is reported, never treated as a zero return.


In [ ]:
display(results["holdout"])
for model in results["selected"].to_dict("records"):
    bt = results["curves"].get(model["pair"])
    if bt is None:
        print(model["pair"], "has unavailable holdout data; see report.")
        continue
    plot_pair(bt, model, CONFIG, RUN_DIR / (model["pair"].replace("/", "_")+"_diagnostics.png"))
    plt.show()


## Accounting and production limitations

At entry, long/short dollar weights are proportional to `(1, -beta)` and gross exposure is 100% of equity. The fitted log-price beta is an elasticity, not a share ratio. Adjusted-price units are held fixed until exit; dollar exposures drift. This is a total-return price approximation, not a broker cash ledger. Adjusted data approximate distributions, including the economic cost on shorts; do not separately subtract dividends again. Production needs raw executable prices, explicit corporate actions, dividend liabilities, cash interest, stock-loan terms and margin accounting.

Transaction costs apply to each leg's actual traded notional on entry and exit. Borrow fees use calendar days and short notional; the default 100 bps/year is a scenario assumption, not a quote. Both mean-reversion and stop decisions use yesterday's z-score; time exits and final scheduled liquidation also pay costs. A z-score stop does not cap realized loss. Daily adjusted close prices cannot model spreads, partial fills, halts, market impact or borrow recalls. Gross exposure can drift beyond its entry value.

The price gate uses median adjusted formation price as a rough screen; production should use contemporaneous raw prices. Data filters and finite-sample model assumptions affect inference. Raw mode does not control multiple testing. Optional corrections address the statistical screen, not repeated strategy tuning or selection by validation Sharpe.

Next steps before deployment: point-in-time membership and sector data; nested rolling formation/validation/trading windows; re-estimation on a fixed schedule with explicit rehedging costs; structural-break monitoring; portfolio exposure and pair-return correlation limits; borrow availability and execution simulations; paper trading and reconciliation. Rolling beta is intentionally not bolted onto a fixed-beta spread without adjusting the accounting.

### References

- [Statsmodels Engle–Granger: null, integration assumptions and residual critical values](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.coint.html)
- [Statsmodels multiple-testing methods](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html)
- [yfinance download API](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html)
- [yfinance project and data-use notes](https://github.com/ranaroussi/yfinance)
- [Public current constituent table](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies)
